In [1]:
from zipfile import ZipFile
def extract(filename):
    input_zip = f"/kaggle/input/competitions/aerial-cactus-identification/{filename}" 
    extract_to = "/kaggle/working/"
    
    with ZipFile(input_zip, "r") as zip_ref:
        zip_ref.extractall(extract_to)
        
    print(f"Extract {filename} complete!")

In [2]:
extract("train.zip")
extract("test.zip")

Extract train.zip complete!
Extract test.zip complete!


In [51]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image

class MyDataset(Dataset):
    def __init__(self, folder_dir, csv_dir, transform=None, is_test=False):
        self.folder_dir = folder_dir
        self.df = pd.read_csv(csv_dir)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx, 0]
        image_dir = os.path.join(self.folder_dir, img_name)
        image = Image.open(image_dir).convert("RGB")

        if self.transform:
            self.transform(image)
            
        label = -1
        if not self.is_test:
            label = int(self.df.iloc[idx, 1])

        return image, label, img_name
    

In [52]:
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

test_transforms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

train_data = MyDataset("/kaggle/working/train/", 
                          "/kaggle/input/competitions/aerial-cactus-identification/train.csv",
                         train_transforms)
test_data = MyDataset("/kaggle/working/test/", 
                          "/kaggle/input/competitions/aerial-cactus-identification/sample_submission.csv",
                         test_transforms, 
                          True)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [53]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

model = model.to(device)

metric = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [54]:
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels, _ in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = metric(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc:.2f}%")

TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'PIL.Image.Image'>

In [55]:
model.eval() # Set to evaluation mode (turns off dropout/batchnorm updates)
predictions = []

# Tell PyTorch not to calculate gradients to save memory and speed things up
with torch.no_grad():
    for images, _, img_names in test_loader:
        images = images.to(device)
        outputs = model(images)
        
        # Get the probability of class 1 (has cactus)
        probabilities = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        
        # Pair up the image name with its prediction
        for img_name, prob in zip(img_names, probabilities):
            predictions.append({'id': img_name, 'has_cactus': prob})

# Convert list of dictionaries to a Pandas DataFrame
submission_df = pd.DataFrame(predictions)

# Save to CSV
submission_df.to_csv('submission.csv', index=False)
print("Submission saved successfully!")

TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'PIL.Image.Image'>